In [9]:
import os
import pandas as pd


In [15]:
def writeFile(path,fileOut):
    f = open(path,'w')
    f.write(fileOut)
    f.close()

def getFile(path):
    f = open(path,'r')
    filein = f.readlines()
    f.close()
    return filein
   
def get_ddl(databse,schema,table,ddl_column_str):
    ddl = """


CREATE OR REPLACE TABLE {table} (
{ddl_column_str},
_FILEDATE TIMESTAMP_NTZ(9),
_FILENAME varchar(128),
_UPLOADDATE TIMESTAMP_NTZ(9),
AUDIT_LOG_ID varchar(128),
_ROW_HASH varchar(128)
    );
    """.format(
        databse=databse,
        schema=schema,
        table=table,
        ddl_column_str=ddl_column_str
    )
    return ddl

def getDbtCode(col_s_string, col_t_string, col_type_data_str, tablename, lz_schema, jconfig, ifMerge):
    DBT_CODE = """
    /************
    Model file generated from SAPAPIUtility 
    *************/
    
    {jconfig}
    
    with source_data as (
        SELECT
        {col_s_string},
        _FILEDATE,
		_FILENAME,
        _UPLOADDATE
        FROM {{{{ source('{lz_schema}', '{tablename}') }}}} AS {tablename}
        {ifMerge}
    ),
    transform_data as (
        SELECT
        {col_t_string},
        _FILEDATE,
		_FILENAME,
        _UPLOADDATE,
		'{{{{ invocation_id }}}}' AS AUDIT_LOG_ID,
        SHA2('hash_me') as _ROW_HASH
        FROM source_data
    ),
    type_data as (
        SELECT
        {col_type_data_str},
        cast(_FILEDATE AS TIMESTAMP_NTZ(9)) AS _FILEDATE,
		cast(_FILENAME as varchar(128)) as _FILENAME,
        cast(_UPLOADDATE as TIMESTAMP_NTZ(9)) as _UPLOADDATE,
        cast(AUDIT_LOG_ID as varchar(128)) as AUDIT_LOG_ID,
        cast(_ROW_HASH as varchar(128)) as _ROW_HASH
        FROM transform_data
    )
    SELECT * FROM type_data
    """.format(
        jconfig=jconfig,
        col_s_string=col_s_string,
        lz_schema=lz_schema,
        tablename=tablename,
        ifMerge=ifMerge,
        col_t_string=col_t_string,
        col_type_data_str=col_type_data_str
    )
    return DBT_CODE    

def getJinjaConfig(strategy, cdsView, unique_keys):
    
    jinjaConfig = """
    {{{{ config(
      materialized='incremental',
      incremental_strategy='{}',
      alias='{}',
      unique_key=['{}'],
      transient=false
    ) }}}} """.format(strategy, cdsView, unique_keys)
    
    return jinjaConfig

def createSchema(df_tables_test,lzSchema,odsSchema,schemaPath):
        df_tables_test['schema'] = """\n      - name: """ + df_tables_test['TABLE_NAME'] + """\n        identifier: """ + df_tables_test['TABLE_NAME'] + """\n        description:  Landing table for """ + df_tables_test['TABLE_NAME'] + """ file. """

        schema_str = """version: 2\nsources:\n  - name: """ + odsSchema + """\n    description: Landing Zone - """ + lzSchema + """ Schema\n    database: "{{ env_var('SNOWFLAKE_LZ') }}"\n    schema: """+ lzSchema + """\n    tables:"""

        for idx, r in df_tables_test.iterrows():
            schema_str += r['schema'] + """
        """   

        df_tables_test['models'] = """\n  - name: """ + df_tables_test['TABLE_NAME'] + """\n    identifier: """ + df_tables_test['TABLE_NAME'] + """\n    description:  Landing table for """ + df_tables_test['TABLE_NAME'] + """ file. """

        schema_str_t ="""\nmodels:"""
        for idx, r in df_tables_test.iterrows():
            schema_str_t += r['models'] + """
        """
        final_schema = schema_str + schema_str_t
        	
        path_create = schemaPath
        filename = "schema.yml"
        path_write = path_create +"\\"+ filename

        # createFile(path_create,filename|)
        writeFile(path_write,final_schema)
        


In [23]:
dbt_path = "C:\\Repo\\EnterpriseCloud\\dbt\\lz_to_ods\\models\\crm_file\\"
csv_path = "C:\\Repo\\EnterpriseCloud\\Notebooks\\schema_07252023.csv"
snow_path = "C:\\Repo\\EnterpriseCloud\\Snow\\K.32049.02__crmfile_ods_ddl.sql" 

In [21]:
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()
unique_tables = df['Table'].unique()
df.head()

,Table,Column,CRM Type,SF Type
0,account,Id,guid,VARCHAR(16777216)
1,account,SinkCreatedOn,dateTime,TIMESTAMP_LTZ(9)
2,account,SinkModifiedOn,dateTime,TIMESTAMP_LTZ(9)
3,account,statecode,int64,"NUMBER(38,0)"
4,account,statuscode,int64,"NUMBER(38,0)"


In [24]:
#Create DDLs - snow file
df["ddl_columns"] = '"' + df["Column"].str.upper() + '" ' + df["SF Type"]
outstr = "USE DATABASE <<ODS>>;\nUSE SCHEMA CRM_FILE;\n"
for table in unique_tables:
    df_ddl = df.loc[df['Table'] == table]
    ddl_columns = []
    for idx,row in df_ddl.iterrows():
        ddl_columns.append(row["ddl_columns"])

    ddl_column_str = ',\n '.join(ddl_columns)
    final_ddl = get_ddl("ODS","CRM_FILE",table,ddl_column_str)
    outstr = outstr + final_ddl + "\n"

writeFile(snow_path,outstr)



In [25]:
#Create dbt models - strategy: merge
dbt_strategy = 'incremental'
for table in unique_tables:
    csvData = df.loc[df['Table'] == table]
    col_t = []
    col_s = []
    col_type_data = []
    for idx,row in csvData.iterrows():
        col_s.append('get_ignore_case(to_object(JSONFIELD),\'' + row["Column"] + '\')::STRING AS "' + row["Column"].upper() + '"')  
        col_type_data.append('CAST("{}" AS {}) AS "{}"'.format(row["Column"].upper(), row["SF Type"], row["Column"].upper()))
        if row["Column"].upper() == 'SINKCREATEDON' or row["Column"].upper() == 'SINKMODIFIEDON':
            col_t.append('TRY_TO_TIMESTAMP("{}", \'MM/DD/YYYY HH:MI:SS PM\') AS "{}"'.format(row["Column"].upper(),row["Column"].upper()))
        else:
            col_t.append('"'+row["Column"].upper()+'"')
               
    col_type_data_str = ', \n \t \t'.join(col_type_data)
    col_t_string = ', \n \t \t'.join(col_t)
    col_s_string = ', \n \t \t'.join(col_s)
    

    if dbt_strategy == 'incremental':
        strategy = 'merge'
        
        ifMerge = """{% if is_incremental() %}WHERE  _UPLOADDATE > IFNULL((SELECT MAX(_UPLOADDATE) FROM {{ this }}),'1970-01-01'){% endif %}
        QUALIFY (ROW_NUMBER() OVER (PARTITION BY get_ignore_case(to_object(JSONFIELD),'Id')::STRING  ORDER BY _FILEDATE DESC,_UPLOADDATE DESC)) = 1"""  

    else:
        strategy = 'insert_overwrite'
        #ifMerge = """where REGEXP_REPLACE(split_part(_filename, '.',  2),'-[^-]*$','') = (SELECT MAX(REGEXP_REPLACE(split_part(_filename, '.',  2),'-[^-]*$','')) from """ + "{{ source('"+ odsSchema + "','" + tableName + "') }})" """"""
        #unique_fields_str = ''
    table =  table.upper()
    #create dbt model jinja config
    jinjaConfig = getJinjaConfig(strategy,table,"ID")
    dbt_model = getDbtCode(col_s_string,col_t_string,col_type_data_str,table,"CRM_FILE",jinjaConfig,ifMerge)
    
    
    filename = table.upper() + ".sql"
    path_write = dbt_path + filename

    writeFile(path_write,dbt_model)
    print('created file',filename)
    
    
        

created file ACCOUNT.sql
created file ANNOTATION.sql
created file BOOKABLERESOURCE.sql
created file BOOKABLERESOURCEBOOKING.sql
created file BOOKABLERESOURCECATEGORY.sql
created file BOOKABLERESOURCECHARACTERISTIC.sql
created file BOOKINGSTATUS.sql
created file BUSINESSUNIT.sql
created file CAMPAIGN.sql
created file CATEGORY.sql
created file CNT_CNT_CONTRACTSUMMARY_ACCOUNT_FAB.sql
created file CNT_CONTRACTPRODUCTGROUP.sql
created file CNT_CONTRACTSCOPEMASTER.sql
created file CNT_CONTRACTSUMMARY.sql
created file CNT_MSDYN_AGREEMENT_MSDYN_CUSTOMERASSET.sql
created file CNT_ORDERNUMBER.sql
created file CNT_ORDERNUMBERGROUP.sql
created file CNT_SERVICEORDERINFORMATION.sql
created file COM_ALARMCODEINFO.sql
created file COM_COUNTRYMASTER.sql
created file COM_PRODUCTLINE.sql
created file COM_PRODUCTMODEL.sql
created file COM_PRODUCTSYSTEMTYPE.sql
created file COM_SERVICEORDERACTIVITIES_TEA.sql
created file COM_SERVICEORDERACTUALMATERIALCOSTS_TEA.sql
created file COM_SERVICEORDERACTUALOTHERCO

In [27]:
#create schema.yml file
df_test = pd.DataFrame(unique_tables, columns=['TABLE_NAME'])
print(df_test)
df_test["TABLE_NAME"] = df_test["TABLE_NAME"].str.upper()
createSchema(df_test,"CRM_FILE","CRM_FILE",dbt_path)

                   TABLE_NAME
0                     account
1                  annotation
2            bookableresource
3     bookableresourcebooking
4    bookableresourcecategory
..                        ...
188            StatusMetadata
189   GlobalOptionSetMetadata
190         OptionSetMetadata
191            TargetMetadata
192         AttributeMetadata

[193 rows x 1 columns]


In [26]:
#append Schema.yml file
df_tables_test = df_test


schema_path = dbt_path + "schema.yml"

existing_schemas = []
#if schema file exists
#look for schema names
#add to list
if os.path.exists(schema_path):
    schema_file = getFile(schema_path)
    for line in schema_file:        
        if 'identifier:' in line:
            cdsName = line.replace("identifier:","")
            cdsName = cdsName.replace("'","")
            cdsName = cdsName.replace(",","")
            cdsName = cdsName.strip()
            existing_schemas.append(cdsName)
#remove any duplicates
existing_schemas = list(dict.fromkeys(existing_schemas))
#convert to dataframe
es = pd.DataFrame({'TABLE_NAME':existing_schemas})
#merge lists
df_tables_test = pd.concat([df_tables_test,es],ignore_index=True)
createSchema(df_tables_test,"CRM_FILE","CRM_FILE",dbt_path)